# ⟁ Artha — Fine-tuning on Google Colab
> Training TinyLlama 1.1B on Artha corpus using LoRA

In [ ]:
# Step 1 — Install dependencies
!pip install -q transformers peft datasets accelerate torch tokenizers

In [ ]:
# Step 2 — Clone your repo
!git clone https://github.com/yourusername/artha.git
%cd artha
!pip install -e . -q

In [ ]:
# Step 3 — Generate corpus
!python3 examples/generate_corpus.py 50000

In [ ]:
# Step 4 — Train tokenizer
!python3 examples/train_tokenizer.py

In [ ]:
# Step 5 — Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB' if torch.cuda.is_available() else '')

In [ ]:
# Step 6 — Fine-tune!
import json
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_DIR = 'data/artha-model'
CORPUS_PATH = 'data/corpus.jsonl'

# Load corpus
pairs = []
with open(CORPUS_PATH) as f:
    for line in f:
        item = json.loads(line)
        pairs.append({'text': f"<|artha|>\n{item['english']}\n<|compress|>\n{item['artha']}<|end|>"})

print(f'Loaded {len(pairs)} pairs')
dataset = Dataset.from_list(pairs)
split = dataset.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(split["train"])} | Eval: {len(split["test"])}')

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({
    'additional_special_tokens': ['<|artha|>', '<|compress|>', '<|end|>']
})
print('Tokenizer loaded. Vocab size:', len(tokenizer))

In [ ]:
# Tokenize
def tokenize(example):
    result = tokenizer(
        example['text'],
        truncation=True,
        max_length=128,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

train_ds = split['train'].map(tokenize, batched=True, remove_columns=['text'])
eval_ds = split['test'].map(tokenize, batched=True, remove_columns=['text'])
print('Tokenization complete.')

In [ ]:
# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map='auto',
)
model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
print('Model loaded.')

In [ ]:
# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Train!
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
    optim='adamw_torch',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
)

print('Starting training...')
trainer.train()

In [ ]:
# Save model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Model saved to', OUTPUT_DIR)

In [ ]:
# Step 7 — Test inference!
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map='auto'
)
trained = PeftModel.from_pretrained(base, OUTPUT_DIR)
trained.eval()

def compress(prompt):
    input_text = f'<|artha|>\n{prompt}\n<|compress|>\n'
    inputs = tokenizer(input_text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = trained.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    return decoded.split('<|compress|>')[-1].split('<|end|>')[0].strip()

test_prompts = [
    'Please summarise this article in 3 bullet points, focus on key facts',
    'Fix the bug in this Python code and explain what was wrong',
    'Write a formal email to a client about the project delay, under 150 words',
    'Compare React and Vue for a beginner, format as table',
    'Explain machine learning to a 10 year old in simple language',
]

print('ARTHA COMPRESSION TEST')
print('=' * 60)
total_en, total_ar = 0, 0
for prompt in test_prompts:
    compressed = compress(prompt)
    en_t = len(prompt.split())
    ar_t = len(compressed.split())
    total_en += en_t
    total_ar += ar_t
    pct = round((1 - ar_t/en_t) * 100)
    print(f'EN ({en_t:2d}t): {prompt}')
    print(f'AR ({ar_t:2d}t): {compressed}')
    print(f'Saved: {pct}%')
    print()

overall = round((1 - total_ar/total_en) * 100)
print(f'Overall compression: {overall}%')

In [ ]:
# Step 8 — Download model from Colab
from google.colab import files
import shutil

shutil.make_archive('artha-model', 'zip', OUTPUT_DIR)
files.download('artha-model.zip')
print('Model downloaded!')